In [1]:
!pip install -q mne

import os
import gc
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from google.colab import drive
from huggingface_hub import hf_hub_download

# 1. ربط Google Drive وتحميل بيانات الاختبار المرجعية
drive.mount('/content/drive')
drive_dir = "/content/drive/MyDrive/Brain_to_Image"

clip_test_embeddings = torch.load(os.path.join(drive_dir, "clip_test_embeddings.pt")).float()
meta = np.load(os.path.join(drive_dir, "image_metadata.npy"), allow_pickle=True).item()
print(f"تم تحميل متجهات صور الاختبار (200 صورة): {clip_test_embeddings.shape}")

# 2. تحميل ومعالجة إشارات الاختبار (sub-01 / ses-01)
print("\nجاري تحميل إشارات الاختبار الخام (raw_eeg_test.npy)...")
test_eeg_path = hf_hub_download(
    repo_id="gasparyanartur/things-eeg2",
    filename="raw-eeg/sub-01/ses-01/raw_eeg_test.npy",
    repo_type="dataset"
)
eeg_dict = np.load(test_eeg_path, allow_pickle=True).item()

info = mne.create_info(ch_names=eeg_dict['ch_names'], sfreq=eeg_dict['sfreq'], ch_types=eeg_dict['ch_types'])
raw = mne.io.RawArray(eeg_dict['raw_eeg_data'], info, verbose=False)
del eeg_dict
gc.collect()

print("جاري الفلترة (1-40 Hz) وتقليل التردد لـ 250 Hz...")
raw.filter(l_freq=1.0, h_freq=40.0, n_jobs=-1, verbose=False)
raw.resample(250, verbose=False)

events = mne.find_events(raw, stim_channel='stim', verbose=False)
epochs = mne.Epochs(raw, events, tmin=-0.2, tmax=0.8, baseline=(None, 0), preload=True, verbose=False)

eeg_array = epochs.get_data()
labels = epochs.events[:, 2]
del raw, epochs
gc.collect()

# 3. تطبيق Trial Averaging (دمج الـ 20 تكرار لكل صورة لتنقية الإشارة)
X_trials = torch.from_numpy(eeg_array).float()
X_trials = (X_trials - X_trials.mean(dim=-1, keepdim=True)) / (X_trials.std(dim=-1, keepdim=True) + 1e-6)

X_test_avg = []
for img_id in range(1, 201):
    mask = (labels == img_id)
    avg_signal = X_trials[mask].mean(dim=0)
    X_test_avg.append(avg_signal)

X_test = torch.stack(X_test_avg, dim=0) # (200, 64, 251)
del eeg_array, X_trials
gc.collect()

print(f"\nتم دمج التكرارات بنجاح! أبعاد إشارات الاختبار النقية: {X_test.shape}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Mounted at /content/drive
تم تحميل متجهات صور الاختبار (200 صورة): torch.Size([200, 768])

جاري تحميل إشارات الاختبار الخام (raw_eeg_test.npy)...


raw-eeg/sub-01/ses-01/raw_eeg_test.npy: reconstructing file:   0%|          |  0.00B /  694MB            

raw-eeg/sub-01/ses-01/raw_eeg_test.npy: downloading bytes:           |  0.00B            

جاري الفلترة (1-40 Hz) وتقليل التردد لـ 250 Hz...

تم دمج التكرارات بنجاح! أبعاد إشارات الاختبار النقية: torch.Size([200, 64, 251])


In [2]:
class EEGEncoder(nn.Module):
    def __init__(self, in_channels=64, embed_dim=768):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Conv1d(128, 256, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Conv1d(256, 512, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(512),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(4),
            nn.Flatten()
        )
        self.projector = nn.Sequential(
            nn.Linear(512 * 4, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(1024, embed_dim)
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, x):
        features = self.encoder(x)
        embeds = self.projector(features)
        return F.normalize(embeds, dim=-1)

print("تم تعريف معمارية EEGEncoder بنجاح!")

تم تعريف معمارية EEGEncoder بنجاح!


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
X_test_gpu = X_test.to(device)
Y_test_gpu = clip_test_embeddings.to(device)

def evaluate_checkpoint(ckpt_name, label):
    model = EEGEncoder().to(device)
    model.load_state_dict(torch.load(os.path.join(drive_dir, ckpt_name), map_location=device))
    model.eval()
    with torch.no_grad():
        preds = model(X_test_gpu)

    sim_matrix = preds @ Y_test_gpu.T
    ranks = sim_matrix.argsort(dim=-1, descending=True)
    gt = torch.arange(200, device=device).unsqueeze(1)

    top1 = (ranks[:, :1] == gt).any(dim=-1).float().mean().item() * 100
    top5 = (ranks[:, :5] == gt).any(dim=-1).float().mean().item() * 100
    top10 = (ranks[:, :10] == gt).any(dim=-1).float().mean().item() * 100
    cos_sim = F.cosine_similarity(preds, Y_test_gpu).mean().item()

    print(f"[{label}] -> Top-1: {top1:.2f}% | Top-5: {top5:.2f}% | Top-10: {top10:.2f}% | Cosine Sim: {cos_sim:.4f}")
    return preds.cpu(), top5

print("=== مقارنة الأداء على 200 صورة اختبار (بعد Trial Averaging) ===")
preds_v1, score_v1 = evaluate_checkpoint("best_eeg_encoder.pth", "الموديل الأول V1")
preds_v2, score_v2 = evaluate_checkpoint("best_eeg_encoder_v2.pth", "الموديل المحسن V2")

# اختيار الموديل الفائز وحفظ متجهاته للخطوة القادمة (توليد الصور بـ Stable Diffusion)
best_preds = preds_v2 if score_v2 >= score_v1 else preds_v1
winner_name = "V2" if score_v2 >= score_v1 else "V1"
torch.save(best_preds, os.path.join(drive_dir, "predicted_test_eeg_embeds.pt"))
print(f"\nتم اعتماد الموديل الفائز ({winner_name}) وحفظ المتجهات المتوقعة لتوليد الصور!")

=== مقارنة الأداء على 200 صورة اختبار (بعد Trial Averaging) ===
[الموديل الأول V1] -> Top-1: 5.50% | Top-5: 12.50% | Top-10: 24.00% | Cosine Sim: 0.0813
[الموديل المحسن V2] -> Top-1: 3.00% | Top-5: 16.50% | Top-10: 27.00% | Cosine Sim: 0.0676

تم اعتماد الموديل الفائز (V2) وحفظ المتجهات المتوقعة لتوليد الصور!
